# Projet Leyenda - Livrable 3
##### *Par Noah.V, Noah.M, Hugo.L*

## **Contexte:**
L'entreprise TouNum est une entreprise qui travaille sur la numérisation de différents type de documents. Ils essayent de moderniser leur architecture pour le tri automatique de documents due au grand nombre de documents fourni par les clients. Faisant appel à nous pour determiner le captionning des images avec légendes de manière automatiques. Nous remarquons deux contraintes, la première est la qualité des images qui diffère par exemple entre chaque image, et trié automatiquement si c'est une photo ou d'autres types de documents.

Le workflow demandé par l'entreprise devra suivre ce modèle la:

![Image demandé](src/workflow.png)

Le projet sera divisé en trois partie:
- La première sera de contrsuire un CNN via des datasets d'images pré étiqueté
- Le deuxième sera d'améliorer la clarté et l'enlever les différents bruits ou flou des images
- Le troisème sera de créer un réseau de neurones qui génère des légendes pour des photographies

Après avoir réalisé les deux premiers livrables, l'objectif est de créer un réseau de neurones qui génère des légendes pour des photographies, en s'appuyant sur le dataset dataset MS COCO. Il sera divisé en 2 partie la premiere via un CNN et la deuxieme sera le RNN pour le captionning. Nous utilisons un modèle pré-entrainé: *Inception V3*

Détail du ***Livrable 3:***
- L'architecture schématique complète du réseau utilisé pour le captioning explicitant le type de CNN utilisé pour les prétraitements.
- Un petit descriptif sur le pré-traitements de images et du texte.
- Le code explicitant l'architecture du CNN et du RNN utilisés dans le captioning.
- L'évolution sous forme de courbes des performances du réseau pendant l'entrainement. Affichage de quelques exemples pour les tests.

## Importation des librairies

In [ ]:
%pip requirements.txt

In [ ]:
# Imports principaux
import tensorflow as tf
import matplotlib.pyplot as plt
import collections
import random
import numpy as np
import time
import json
import math
from pathlib import Path
from tqdm import tqdm

## Etape 1: Architecture schématique du modèle.

Voici l'architecture globale du modèle de captioning :
![Image demandé](src/Captioning_model.jpg)

Fonctionnement du modèle :

Le modèle prend en entrée une image et sa légende associée. L’image est prétraitée, puis transmise à InceptionV3, utilisé avec des poids gelés pour extraire ses caractéristiques visuelles. Le CNN Encoder transforme ensuite ces caractéristiques en 64 vecteurs de 256 valeurs. En parallèle, la légende est préparée, convertie en tokens et complétée par du padding.

À chaque étape, le mécanisme d’attention utilise les caractéristiques de l’image et l’état caché précédent du décodeur pour calculer un contexte visuel. Le décodeur, basé sur un LSTM ou un GRU, utilise ce contexte et le token précédent pour produire les scores du prochain token. Son nouvel état caché est conservé pour l’étape suivante.

Pendant l’entraînement, les scores sont comparés au token cible de la légende de référence afin de calculer la loss et d’ajuster les paramètres entraînables du modèle. Le processus se répète sur les différents tokens de la légende.

Voici le détail de l’architecture de l’encodeur :

![Image demandé](src/Captioning_model-CNN.jpg)

Fonctionnement du CNN Encoder :

Le CNN Encoder reçoit les caractéristiques extraites par InceptionV3, sous la forme de 64 vecteurs de 2 048 valeurs. Chaque vecteur correspond à une position dans la grille de l’image. La couche Dense applique la même projection à chacun de ces vecteurs pour passer de 2 048 à 256 valeurs. L’activation ReLU remplace ensuite les valeurs négatives par zéro. On obtient donc 64 vecteurs de 256 valeurs : la représentation est réduite, mais les différentes positions de l’image sont conservées pour le mécanisme d’attention.

Voici le détail de l’architecture du décodeur :

![Image demandé](src/Captioning_model-RNN.jpg)

Fonctionnement du RNN Decoder (GRU):


Le décodeur reçoit les caractéristiques visuelles encodées, le token précédent et l’état caché de l’étape précédente. Le token est transformé par la couche Embedding en un vecteur de 256 valeurs. En parallèle, l’attention de Bahdanau utilise les caractéristiques de l’image et l’état caché précédent pour pondérer les 64 positions et construire un contexte visuel de 256 valeurs. Ces deux vecteurs sont concaténés pour former l’entrée du GRU, soit 512 valeurs. 

Le GRU produit une sortie et un nouvel état caché de 512 valeurs, réutilisé par l’attention au tour suivant. Dans cette implémentation, l’état précédent est transmis à l’attention, mais pas directement au GRU comme état initial. La sortie passe ensuite par une couche Dense de 512 neurones, une activation ReLU et un Reshape qui adapte sa forme avant la dernière couche. 

La couche Dense finale produit 5 001 scores, appelés logits, soit un par identifiant du vocabulaire de sortie. Pendant l’entraînement, ces scores sont comparés au token cible pour calculer la loss. Lors de la génération, le code sélectionne le prochain token par échantillonnage à partir des logits, puis le réutilise comme entrée à l’étape suivante, jusqu’au token de fin ou à la longueur maximale autorisée.


## Etape 2 : Chargement et Pré-traitement des images


### Importation du dataset et annotation COCO

In [ ]:
# Sélection déterministe par image : toutes les captions d'une image restent ensemble.
MAX_IMAGES = {"train": 30000, "validation": 10000}
SUBSET_SEED = 42
DATA_DIR = Path("data")

ANNOTATION_FILES = {
    "train": DATA_DIR / "annotations" / "captions_train2014.json",
    "validation": DATA_DIR / "annotations" / "captions_val2014.json",
}
IMAGE_DIRS = {
    "train": DATA_DIR / "train2014",
    "validation": DATA_DIR / "val2014",
}

def select_image_ids(annotation_data, split):
    """Sélectionne au plus MAX_IMAGES identifiants uniques avec une seed fixe."""
    image_ids = sorted({a["image_id"] for a in annotation_data["annotations"]})
    rng = random.Random(SUBSET_SEED)
    return set(rng.sample(image_ids, min(MAX_IMAGES[split], len(image_ids))))

def load_split(split):
    with ANNOTATION_FILES[split].open("r", encoding="utf-8") as f:
        annotation_data = json.load(f)
    selected_ids = select_image_ids(annotation_data, split)
    prefix = "train" if split == "train" else "val"
    image_to_captions = collections.defaultdict(list)
    for annotation in annotation_data["annotations"]:
        if annotation["image_id"] in selected_ids:
            filename = f"COCO_{prefix}2014_{annotation['image_id']:012d}.jpg"
            image_path = IMAGE_DIRS[split] / filename
            image_to_captions[image_path].append(
                f"<start> {annotation['caption']} <end>"
            )
    image_paths = sorted(image_to_captions)
    captions, image_names = [], []
    for image_path in image_paths:
        captions.extend(image_to_captions[image_path])
        image_names.extend([str(image_path)] * len(image_to_captions[image_path]))
    print(f"{split}: {len(image_paths)} images uniques, {len(captions)} captions")
    return image_names, captions

img_name_train, train_captions = load_split("train")
img_name_val, val_captions = load_split("validation")

validation_captions = val_captions


train : 82,783 images, 414,113 légendes
  images introuvables sur disque : 0
  images sans légende            : 0
validation : 40,504 images, 202,654 légendes
  images introuvables sur disque : 0
  images sans légende            : 0
IDs communs train/validation : 0
ID image : 57870
Chemin : Data\train2014\COCO_train2014_000000057870.jpg
- A restaurant has modern wooden tables and chairs.
- A long restaurant table with rattan rounded back chairs.
- a long table with a plant on top of it surrounded with wooden chairs 
- A long table with a flower arrangement in the middle for meetings
- A table is adorned with wooden chairs with blue accents.


Dans cette partie, nous chargeons les annotations des jeux d'entraînement et de validation, qui se trouvent dans le dossier data. Nous gardons au maximum 30 000 images pour l'entraînement et 10 000 pour la validation. La sélection est faite avec une graine fixe, ce qui nous permet de retrouver les mêmes images à chaque exécution.

Quand une image est choisie, nous gardons toutes ses légendes. Nous ajoutons ensuite "start" et "end" au début et à la fin de chaque légende, pour que le modèle sache où elle commence et où elle se termine.

À la fin, nous obtenons deux listes qui se correspondent : une avec les chemins des images, l'autre avec leurs légendes. Comme une image a plusieurs légendes, son chemin apparaît plusieurs fois. Le nombre d'images uniques et de légendes retenues s'affiche pour contrôler le résultat.

Ensuite, nous passons au prétraitement des images.

In [ ]:
# Extraction un fichier par image, stockage float16.
FEATURES_DIR = Path("features/inception_grid_fp16")
EXTRACTION_BATCH_SIZE = 64
FEATURE_SHAPE = (64, 2048)
FEATURE_PATHS = {}
for split, image_names in (("train", img_name_train), ("validation", img_name_val)):
    for name in sorted(set(image_names)):
        image_id = int(Path(name).stem.rsplit("_", 1)[-1])
        FEATURE_PATHS[name] = FEATURES_DIR / split / f"{image_id}.npy"

image_model = tf.keras.applications.InceptionV3(include_top=False, weights="imagenet")
image_model.trainable = False
image_features_extract_model = tf.keras.Model(image_model.input, image_model.layers[-1].output)

def load_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (299, 299))
    img = tf.keras.applications.inception_v3.preprocess_input(img)
    return img, image_path

def valid_feature_file(path):
    if not path.is_file():
        return False
    try:
        values = np.load(path, allow_pickle=False)
        return (values.shape == FEATURE_SHAPE
                and values.dtype in (np.dtype("float16"), np.dtype("float32"))
                and bool(np.isfinite(values).all()))
    except (OSError, ValueError, EOFError):
        return False

def save_feature_file(path, values):
    values = np.asarray(values, dtype=np.float16)
    if values.shape != FEATURE_SHAPE or not np.isfinite(values).all():
        raise ValueError(f"Features invalides : {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp.npy")
    np.save(temporary, values, allow_pickle=False)
    temporary.replace(path)

def extract_features(image_paths, dataset_name):
    pending = []
    reused = migrated = 0
    for name in tqdm(sorted(set(image_paths)), desc=f"Vérification {dataset_name}"):
        destination = FEATURE_PATHS[name]
        if valid_feature_file(destination):
            reused += 1
            continue
        legacy = Path(name + ".npy")
        if valid_feature_file(legacy):
            save_feature_file(destination, np.load(legacy, allow_pickle=False))
            migrated += 1
        else:
            pending.append(name)
    print(f"{dataset_name} : {reused} présentes, {migrated} reprises depuis les anciens .npy, {len(pending)} à extraire.")
    if not pending:
        return
    image_dataset = tf.data.Dataset.from_tensor_slices(pending)
    image_dataset = image_dataset.map(
        load_image, num_parallel_calls=tf.data.AUTOTUNE
    ).batch(EXTRACTION_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    total = math.ceil(len(pending) / EXTRACTION_BATCH_SIZE)
    for images, paths in tqdm(image_dataset, total=total, desc=f"Extraction {dataset_name}"):
        features = image_features_extract_model(images, training=False)
        features = tf.reshape(features, (tf.shape(features)[0], 64, 2048))
        for values, name in zip(features.numpy(), paths.numpy()):
            save_feature_file(FEATURE_PATHS[name.decode("utf-8")], values)

extract_features(img_name_train, "train")
extract_features(img_name_val, "validation")


Entrée : (None, 299, 299, 3)
Sortie : (None, 8, 8, 2048)


Dans cette partie, nous utilisons InceptionV3, préentraîné sur ImageNet, pour extraire les caractéristiques visuelles des images. Nous retirons sa couche finale de classification avec include_top=False, puis nous gelons ses poids : ce modèle n’est pas réentraîné ici. C’est le principe du transfer learning.

Avant l’extraction, chaque image est lue, redimensionnée en 299 × 299 et normalisée avec *preprocess_input* d’InceptionV3. La dernière couche du réseau produit une grille de 8 × 8 positions, avec 2 048 valeurs par position. Nous la transformons en 64 vecteurs de 2 048 valeurs.

Ces caractéristiques sont enregistrées dans un fichier .npy par image, nommé avec son identifiant, en float16 pour limiter l’espace occupé. Avant de lancer le calcul, le code vérifie si un fichier valide existe déjà : seules les images manquantes passent par InceptionV3. Cela évite de recalculer les mêmes caractéristiques à chaque exécution et réduit le temps de calcul pendant l’entraînement.

Ensuite, ces fichiers sont rechargés, convertis en float32, puis envoyés à l’encodeur CNN.

In [ ]:
# Tokenizer 
def calc_max_length(tensor):
    return max(len(t) for t in tensor)

top_k = 5000
tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=top_k, oov_token="<unk>",
    filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ '
)
tokenizer.fit_on_texts(train_captions)
tokenizer.word_index["<pad>"] = 0
tokenizer.index_word[0] = "<pad>"

train_seqs = tokenizer.texts_to_sequences(train_captions)
max_length = calc_max_length(train_seqs)
cap_train = tf.keras.preprocessing.sequence.pad_sequences(train_seqs, padding="post")
val_seqs = tokenizer.texts_to_sequences(val_captions)
cap_val = tf.keras.preprocessing.sequence.pad_sequences(
    val_seqs, maxlen=max_length, padding="post"
)
print("Token <start> :", tokenizer.word_index.get("<start>"))
print("Token <end>   :", tokenizer.word_index.get("<end>"))


In [ ]:
# verification image
print(f"Train : {len(set(img_name_train))} images uniques, {len(cap_train)} captions")
print(f"Validation : {len(set(img_name_val))} images uniques, {len(cap_val)} captions")


In [ ]:
#hyperparametre
BATCH_SIZE = 128
BUFFER_SIZE = 1000
embedding_dim = 256
units = 512
vocab_size = top_k + 1
num_steps = max(1, math.ceil(len(img_name_train) / BATCH_SIZE))
num_validation_steps = max(1, math.ceil(len(img_name_val) / BATCH_SIZE))
features_shape = 2048
attention_features_shape = 64
USE_CACHE = True  # True pour activer un cache disque local.
EPOCHS = 10
patience = 3
min_delta = 0.001
wait = 0

Dans cette partie, nous préparons les légendes. Le tokenizer est appris uniquement sur les légendes d'entraînement. Il garde les 5 000 mots les plus fréquents, et les autres sont remplacés par le jeton "unk". Nous ne filtrons pas les symboles < et >, pour que "start" et "end" soient conservés. Nous ajoutons aussi un jeton "pad" d'indice 0, qui sert à compléter les séquences.

Chaque légende est ensuite transformée en suite d'entiers. Pour que toutes les séquences aient la même longueur, nous complétons par des zéros à la fin. La longueur retenue est celle de la plus longue légende d'entraînement, et nous l'appliquons aussi à la validation. Nous affichons ensuite le nombre d'images uniques et de légendes pour contrôler le résultat.

Enfin, nous définissons les hyperparamètres du modèle :

- Les batches contiennent 128 exemples, et les données sont mélangées avec un buffer de 1 000 éléments.

- Les mots sont représentés par des vecteurs de taille 256, et le GRU comme l'attention utilisent 512 unités.

- La taille du vocabulaire est de 5 001 : les 5 000 mots retenus plus le jeton de remplissage.

- Les features d'image ont 64 positions de 2 048 valeurs.

- L'entraînement dure au plus 10 époques, avec un arrêt anticipé si la validation ne s'améliore pas d'au moins 0,001 pendant 3 époques consécutives. La variable wait compte ces époques.

In [ ]:
def map_func(img_name, cap):
    img_tensor = np.load(FEATURE_PATHS[img_name.decode("utf-8")], allow_pickle=False)
    if img_tensor.shape != FEATURE_SHAPE:
        raise ValueError(f"Forme de features incorrecte : {img_tensor.shape}")
    return img_tensor.astype(np.float32), cap

def make_feature_dataset(image_names, captions, cache_path=None, shuffle_data=False):
    ds = tf.data.Dataset.from_tensor_slices((image_names, captions))
    ds = ds.map(
        lambda name, cap: tf.numpy_function(
            map_func, [name, cap], [tf.float32, tf.int32]
        ),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    if cache_path is not None:
        ds = ds.cache(cache_path)
    if shuffle_data:
        ds = ds.shuffle(BUFFER_SIZE, seed=SUBSET_SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

dataset = make_feature_dataset(
    img_name_train, cap_train, "./tf_cache_features_fp16_v1_train" if USE_CACHE else None, True
)
validation_dataset = make_feature_dataset(
    img_name_val, cap_val, "./tf_cache_features_fp16_v1_validation" if USE_CACHE else None, False
)


Dans cette partie, nous construisons les datasets d'entraînement et de validation. La fonction *map_func()* charge les features d'une image depuis son fichier .npy, vérifie qu'elles ont la forme (64, 2048), puis les convertit en float32 et les renvoie avec la légende associée.

La fonction *make_feature_dataset()* applique cette fonction à chaque exemple, puis regroupe les données en batches de 128 avec préchargement. Pour l'entraînement, les exemples sont mélangés à chaque époque avec une graine fixe. La validation n'est pas mélangée.

Nous utilisons aussi un cache sur disque, pour éviter de recharger les fichiers .npy à chaque époque.

## Etape 3: Architecture avec CNN et RNN

In [ ]:
class CNN_Encoder(tf.keras.Model):
    # Comme les images sont déjà prétraités par InceptionV3 est représenté sous forme compacte
    # L'encodeur CNN ne fera que transmettre ces caractéristiques à une couche dense
    def __init__(self, embedding_dim):
        super(CNN_Encoder, self).__init__()
        # forme après fc == (batch_size, 64, embedding_dim)
        self.fc = tf.keras.layers.Dense(embedding_dim)

    def call(self, x):
        x = self.fc(x)
        x = tf.nn.relu(x)
        return x

2000


Ici, nous créons notre encodeur CNN. Il reçoit les features extraites par InceptionV3, sous la forme de 64 vecteurs de 2 048 valeurs. Une couche Dense réduit chaque vecteur à 256 valeurs, puis une activation ReLU est appliquée. Nous conservons les 64 positions de l'image, ce qui permet ensuite à l'attention de se concentrer sur les zones les plus utiles pour la prédiction.

In [ ]:
class BahdanauAttention(tf.keras.Model):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, features, hidden):
        # Ajoute un axe temporel à l'état caché
        hidden_with_time_axis = tf.expand_dims(hidden, 1)

        attention_hidden_layer = tf.nn.tanh(
            self.W1(features) + self.W2(hidden_with_time_axis)
        )

        # Score non normalisé pour chaque caractéristique de l'image
        score = self.V(attention_hidden_layer)

        attention_weights = tf.nn.softmax(score, axis=1)

        # Somme pondérée des caractéristiques de l'image
        context_vector = attention_weights * features
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

Ici, nous créons le mécanisme d'attention de Bahdanau. Il combine les features de l'image et l'état caché du décodeur pour calculer un score pour chacune des 64 zones de l'image. Un softmax transforme ces scores en poids d'attention, dont la somme vaut 1. Le vecteur de contexte est la somme des features pondérée par ces poids : il résume les zones les plus utiles pour prédire le mot suivant.

In [ ]:
class RNN_Decoder(tf.keras.Model):
    def __init__(self, embedding_dim, units, vocab_size):
        super(RNN_Decoder, self).__init__()
        self.units = units

        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(
            self.units,
            return_sequences=True,
            return_state=True,
            recurrent_initializer='glorot_uniform',
            dropout=0.3,           # Dropout sur les entrées
            recurrent_dropout=0.3
        )
        self.fc1 = tf.keras.layers.Dense(self.units)
        self.fc2 = tf.keras.layers.Dense(vocab_size)
        self.attention = BahdanauAttention(self.units)

    def call(self, x, features, hidden):
        context_vector, attention_weights = self.attention(features, hidden)

        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

        output, state = self.gru(x)
        y = self.fc1(output)
        y = tf.nn.relu(y)

        # Utiliser la dimension de y, et non celle de x.
        y = tf.reshape(y, (-1, y.shape[2]))
        y = self.fc2(y)

        return y, state, attention_weights

    def reset_state(self, batch_size):
        return tf.zeros((batch_size, self.units))

Dans notre architecture, le décodeur RNN utilise un GRU pour générer la légende mot par mot. Le mot précédent est transformé en vecteur par une couche d'embedding, puis concaténé au vecteur de contexte fourni par l'attention. Le GRU, régularisé par un dropout de 0,3, traite cette entrée avec son état précédent. Deux couches Dense successives (la première suivie d'une ReLU) produisent ensuite un score pour chaque mot du vocabulaire, ce qui permet de prédire le mot suivant. Le décodeur renvoie aussi le nouvel état et les poids d'attention.

In [ ]:
# Création de l'encodeur
encoder = CNN_Encoder(embedding_dim)

# Création du décodeur
decoder = RNN_Decoder(embedding_dim, units, vocab_size)


Ici, nous instancions l'encodeur et le décodeur. L'encodeur est créé avec la dimension d'embedding choisie. Le décodeur reçoit en plus le nombre d'unités du GRU et la taille du vocabulaire, qui détermine le nombre de mots possibles en sortie.

In [ ]:
# Optimiseur ADAM
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

# La fonction de perte
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)

    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask

    return tf.reduce_mean(loss_)

Ici, nous définissons l'optimiseur Adam, avec un taux d'apprentissage de 1e-4, ainsi que la fonction de perte. Il s'agit d'une entropie croisée catégorielle qui compare le mot réel aux scores prédits pour chaque mot du vocabulaire. Un masque met à zéro la perte des positions de padding, afin que seuls les vrais mots de la légende influencent l'apprentissage. La perte est ensuite moyennée.

In [ ]:
# Checkpoint du meilleur modèle (un seul fichier conservé).
ckpt = tf.train.Checkpoint(encoder=encoder, decoder=decoder, optimizer=optimizer)
best_ckpt_manager = tf.train.CheckpointManager(
    ckpt, "./checkpoints/best", max_to_keep=1
)

# L'entraînement démarre à zéro ; le meilleur checkpoint sera restauré à la fin.
start_epoch = 0
best_val_loss = float("inf")

Ici, nous configurons la sauvegarde du meilleur modèle : un seul checkpoint (encodeur, décodeur, optimiseur) est conservé, et il sera restauré à la fin de l'entraînement.

# entrainement et test

In [ ]:
loss_plot = []
validation_loss_plot = []

@tf.function
def train_step(img_tensor, target):
    loss = tf.constant(0.0)
    hidden = decoder.reset_state(batch_size=tf.shape(target)[0])
    dec_input = tf.fill([tf.shape(target)[0], 1], tokenizer.word_index["<start>"])
    with tf.GradientTape() as tape:
        features = encoder(img_tensor)
        for i in range(1, target.shape[1]):
            predictions, hidden, _ = decoder(dec_input, features, hidden)
            loss += loss_function(target[:, i], predictions)
            dec_input = tf.expand_dims(target[:, i], 1)
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return loss, loss / tf.cast(target.shape[1], tf.float32)

@tf.function
def validation_step(img_tensor, target):
    loss = tf.constant(0.0)
    hidden = decoder.reset_state(batch_size=tf.shape(target)[0])
    dec_input = tf.fill([tf.shape(target)[0], 1], tokenizer.word_index["<start>"])
    features = encoder(img_tensor)
    for i in range(1, target.shape[1]):
        predictions, hidden, _ = decoder(dec_input, features, hidden)
        loss += loss_function(target[:, i], predictions)
        dec_input = tf.expand_dims(target[:, i], 1)
    return loss / tf.cast(target.shape[1], tf.float32)


Ici, nous définissons les étapes d'entraînement et de validation. Pour chaque batch, le décodeur génère la légende mot par mot à partir du jeton "start", en recevant à chaque étape le vrai mot précédent (teacher forcing). Les pertes de chaque mot sont cumulées. Pendant l'entraînement, les gradients sont calculés puis appliqués par l'optimiseur pour mettre à jour l'encodeur et le décodeur. La validation calcule la même perte, sans mise à jour des poids. Les listes *loss_plot* et *validation_loss_plot* conservent les pertes de chaque époque.

In [ ]:
for epoch in range(start_epoch, EPOCHS):
    start = time.time()
    total_loss = tf.constant(0.0)
    for batch, (img_tensor, target) in enumerate(dataset):
        batch_loss, step_loss = train_step(img_tensor, target)
        total_loss += step_loss
        if batch % 100 == 0:
            print(f"Epoch {epoch + 1} Batch {batch} Loss {batch_loss.numpy() / int(target.shape[1]):.4f}")

    total_validation_loss = tf.constant(0.0)
    for validation_img_tensor, validation_target in validation_dataset:
        total_validation_loss += validation_step(validation_img_tensor, validation_target)
    epoch_loss = total_loss / num_steps
    validation_loss = total_validation_loss / num_validation_steps
    loss_plot.append(float(epoch_loss.numpy()))
    validation_loss_plot.append(float(validation_loss.numpy()))

    current_val = float(validation_loss.numpy())
    if current_val < best_val_loss - min_delta:
        best_val_loss = current_val
        wait = 0
        best_ckpt_manager.save()
        print("Meilleur checkpoint sauvegardé.")
    else:
        wait += 1
    print(f"Epoch {epoch + 1} Loss {float(epoch_loss.numpy()):.6f}")
    print(f"Epoch {epoch + 1} Validation Loss {current_val:.6f}")
    print(f"Temps pour l'époque : {time.time() - start:.1f} s")
    if wait >= patience:
        print("Early stopping déclenché.")
        break

# Restaurer le meilleur état connu après l'entraînement.
if best_ckpt_manager.latest_checkpoint:
    ckpt.restore(best_ckpt_manager.latest_checkpoint).expect_partial()
    print("Meilleur checkpoint restauré :", best_ckpt_manager.latest_checkpoint)


Ici, nous lançons l'entraînement. À chaque époque, le modèle est entraîné sur tous les batchs, puis évalué sur le jeu de validation. Quand la perte de validation s'améliore, le checkpoint est sauvegardé. Si elle ne progresse plus pendant plusieurs époques, un early stopping interrompt l'entraînement. Enfin, le meilleur checkpoint est restauré.

### Validation

In [ ]:
def evaluate(image):
    attention_plot = np.zeros((max_length, attention_features_shape))

    hidden = decoder.reset_state(batch_size=1)

    temp_input = tf.expand_dims(load_image(image)[0], 0)
    img_tensor_val = image_features_extract_model(temp_input)
    img_tensor_val = tf.reshape(img_tensor_val, (img_tensor_val.shape[0], -1, img_tensor_val.shape[3]))

    features = encoder(img_tensor_val)

    dec_input = tf.expand_dims([tokenizer.word_index['<start>']], 0)
    result = []

    for i in range(max_length):
        predictions, hidden, attention_weights = decoder(dec_input, features, hidden)

        attention_plot[i] = tf.reshape(attention_weights, (-1, )).numpy()

        predicted_id = tf.random.categorical(predictions, 1)[0][0].numpy()
        result.append(tokenizer.index_word[predicted_id])

        if tokenizer.index_word[predicted_id] == '<end>':
            return result, attention_plot

        dec_input = tf.expand_dims([predicted_id], 0)

    attention_plot = attention_plot[:len(result), :]
    return result, attention_plot

# Fonction permettant la représentation de l'attention au niveau de l'image
def plot_attention(image, result, attention_plot):
    temp_image = np.array(Image.open(image))

    fig = plt.figure(figsize=(10, 10))

    len_result = len(result)
    for l in range(len_result):
        temp_att = np.resize(attention_plot[l], (8, 8))
        ax = fig.add_subplot(len_result//2, len_result//2, l+1)
        ax.set_title(result[l])
        img = ax.imshow(temp_image)
        ax.imshow(temp_att, cmap='gray', alpha=0.6, extent=img.get_extent())

    plt.tight_layout()
    plt.show()

# Affichage de quelques annotations dans le jeu de test
rid = np.random.randint(0, len(img_name_val))
image = img_name_val[rid]
real_caption = ' '.join([tokenizer.index_word[i] for i in cap_val[rid] if i not in [0]])
result, attention_plot = evaluate(image)

print ('Real Caption:', real_caption)
print ('Prediction Caption:', ' '.join(result))
plot_attention(image, result, attention_plot)

Ici, nous testons le modèle sur une image du jeu de test. La fonction *evaluate* extrait les features de l'image, puis génère la légende mot par mot à partir de "start", en tirant chaque mot au hasard selon les scores du décodeur, jusqu'à "end". Elle enregistre aussi les poids d'attention. Enfin, nous comparons la vraie légende à la légende prédite, et plot_attention superpose à l'image les zones sur lesquelles le modèle s'est concentré pour chaque mot.